# Forecast comparison (CONUS) — Met Office, ERA5, GFS and MERRA-2 against Contrails.org and Google

Compares four PCR-derived forecast arms (Met Office, ERA5, GFS, MERRA-2) against the existing Contrails.org and Google arms, over CONUS. Includes the real bootstrap confidence-interval computation for all four new arms (helpers duplicated per this repo's standalone-notebook convention), followed by three analysis sections built on top of it.

In [ ]:
import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm
from tqdm import tqdm

# Helper functions, colors, etc

In [ ]:
def open_dfs(path):
    """Open and concatenate all dataframes in directory."""
    df_list = [pd.read_parquet(f) for f in tqdm(sorted(glob.glob(f"data/{path}/*.pq")), desc=path)]
    return pd.concat(df_list, axis="index")

In [ ]:
def penalty(df, grouping_key):
    """Compute flight distance penalty."""
    with np.errstate(invalid="ignore", divide="ignore"):
        return df.groupby(grouping_key)[["adsb_dist_in_forecast_pcr", "adsb_dist"]].apply(
            lambda x: x["adsb_dist_in_forecast_pcr"].sum() / x["adsb_dist"].sum()
        )

In [ ]:
def hit_rate(df, grouping_key):
    """Compute hit rate."""
    with np.errstate(invalid="ignore", divide="ignore"):
        return df.groupby(grouping_key)[
            ["observed_pcr_area_in_forecast_pcr", "observed_pcr_area"]
        ].apply(lambda x: x["observed_pcr_area_in_forecast_pcr"].sum() / x["observed_pcr_area"].sum())

In [ ]:
def jackknife(df, resampling_key, statistic):
    """Compute jacknife statistics."""
    groups = df.groupby(resampling_key)
    keys = groups.keys.unique()
    indices = groups.indices

    jn_list = []
    for drop in tqdm(keys, desc="jackknife"):
        idx = np.concatenate([indices[key] for key in keys if key != drop])
        subset = df.iloc[idx]
        jn_list.append(statistic(subset))

    jn = pd.concat(jn_list, axis="columns")
    return jn

In [ ]:
def bootstrap(df, resampling_key, num_iterations, statistic, alpha, rng):
    """Compute bootstrap confidence intervals."""
    groups = df.groupby(resampling_key)
    keys = groups.keys.unique()
    indices = groups.indices

    theta_star_list = []
    for i in tqdm(range(num_iterations), desc="bootstrap"):
        samples = rng.choice(keys, keys.size, replace=True)
        idx = np.concatenate([indices[key] for key in samples])
        resampled = df.iloc[idx]
        theta_star_list.append(statistic(resampled))

    theta_star = pd.concat(theta_star_list, axis="columns")
    theta = statistic(df)
    z0 = (theta_star < theta.values.reshape((-1, 1))).mean(axis="columns").map(norm.ppf)
    jn = jackknife(df, resampling_key, statistic)
    jn_bar = np.mean(jn.values, axis=1, keepdims=True)
    a = ((jn - jn_bar) ** 3).sum(axis="columns") / (
        6.0 * ((jn - jn_bar) ** 2).sum(axis="columns") ** 1.5
    )

    z1 = norm.ppf(alpha / 2.0)
    z2 = norm.ppf(1.0 - alpha / 2.0)
    q1 = (z0 + (z0 + z1) / (1.0 - a * (z0 + z1))).apply(norm.cdf)
    q2 = (z0 + (z0 + z2) / (1.0 - a * (z0 + z2))).apply(norm.cdf)

    # A group with zero observations across the entire window (e.g. sparse GRUAN
    # coverage in a short season) makes `theta`/`theta_star` NaN via 0/0, which
    # otherwise propagates into out-of-[0,1]/NaN percentiles and crashes
    # `.quantile()`. Clip in-range and fall back to a NaN CI (no data -> no CI,
    # not a crash) rather than raising -- a no-op whenever theta is well-defined,
    # so this changes nothing for any group that already had real observations.
    q1 = q1.clip(0.0, 1.0)
    q2 = q2.clip(0.0, 1.0)

    def _quantile_or_nan(x):
        if not (np.isfinite(q1.loc[x.name]) and np.isfinite(q2.loc[x.name])):
            return pd.Series([np.nan, np.nan])
        return x.quantile([q1.loc[x.name], q2.loc[x.name]]).reset_index(drop=True)

    return theta_star.apply(_quantile_or_nan, axis="columns")

In [ ]:
exo_black = "#161a26"
tropo_blue = "#1093ff"
solar_orange = "#f26400"
gray50 = "#939598"

# Met Office / ERA5 / GFS / MERRA-2 PCR arms (CONUS, Sep–Dec 2024)

Full window: 1 Sep – 31 Dec 2024, CONUS, RHi threshold swept 0.70–1.30 in 0.05 steps (13 values, spatial buffer fixed at 0), lead time shortest available (`metoffice` explicitly cycles T+0→T+5; `gfs` gets the same behaviour for free from `GFSForecast`'s forecast-time flooring; `era5` has no lead time at all), regridded onto the existing benchmark grid. The PCR family (`metoffice`/`era5`/`gfs`/`merra2`) covers 14 flight levels (FL310–440); `contrails-org`/`google` natively cover FL270–440 but are restricted to the same FL310–440 below so the cross-family comparison is scored on a matched observation sample.

Four windows appear below: **sepdec** is the full window (N=2928 hourly timesteps). **sepdec-era5n** is restricted to `era5`'s exact completed timestamp subset — `era5`'s progressive data-gathering run was deliberately stopped mid-pass, not a bug — used for the four-arm preview plot and to measure `metoffice`'s own subsampling error at that N. **sepdec-gfsn** is restricted to `gfs`'s exact completed timestamp subset (N=300, a deliberately capped sample per this project's N≈300-for-new-sources policy, not a partial/interrupted run) — used for the three-way PCR-family preview (Plot C). **sepdec-merra2n** is restricted to `merra2`'s exact completed timestamp subset (N≈300, same N≈300-for-new-sources policy, drawn from MERRA-2's native 3-hourly synoptic times rather than the hourly grid every other arm uses — see `preprocess_merra2.py`'s module docstring) — used for the four-way PCR-family preview (Plot D).

**Bootstrap resamples: n=2,000, not the 10,000 used elsewhere in this notebook.** Measured directly (small-n extrapolation, not guessed): one full-N combo at n=10,000 costs ~47 min, one era5-N combo ~15 min — ~3.3 hours sequentially across the original 7 combos. n=2,000 is still a standard sample size for percentile-style BCa CIs and cuts this to ~40 minutes, chosen for a reporting deadline. The gfs-matched-N tier adds 5 more combos (metoffice/era5/contrails-org/google restricted to gfs's N, plus gfs's own). The merra2-matched-N tier below adds 6 further combos (metoffice/era5/gfs/contrails-org/google restricted to merra2's N, plus merra2's own) — each combo is the same shape as an era5-N one, so expect a comparable per-combo cost. Re-run at n=10,000 later if tighter CIs are needed — nothing else in this section depends on the choice of n.

In [ ]:
import os
import sys

sys.path.insert(0, "../..")  # repo root, for contrailbench (not pip-installed)
from contrailbench import pcr

D1_TIMES = pd.date_range("2024-09-01 00:00", "2024-12-31 23:00", freq="1h").to_pydatetime().tolist()
D1_TIMES_SET = set(D1_TIMES)

era5_forecast_dir = "data/era5"
era5_existing = [f"{era5_forecast_dir}/{name}" for name in os.listdir(era5_forecast_dir)]
era5_incomplete = set(
    pcr.pending_times(D1_TIMES, pcr.PCR_FLIGHT_LEVELS, era5_existing, era5_forecast_dir)
)
ERA5_TIMES = sorted(t for t in D1_TIMES if t not in era5_incomplete)
ERA5_TIMES_SET = set(ERA5_TIMES)
ERA5_N = len(ERA5_TIMES)
print(
    f"era5: {ERA5_N}/{len(D1_TIMES)} timestamps complete "
    f"({100 * ERA5_N / len(D1_TIMES):.1f}% of window)"
)

In [ ]:
gfs_forecast_dir = "data/gfs"
gfs_existing = [f"{gfs_forecast_dir}/{name}" for name in os.listdir(gfs_forecast_dir)]
gfs_incomplete = set(
    pcr.pending_times(D1_TIMES, pcr.PCR_FLIGHT_LEVELS, gfs_existing, gfs_forecast_dir)
)
GFS_TIMES = sorted(t for t in D1_TIMES if t not in gfs_incomplete)
GFS_TIMES_SET = set(GFS_TIMES)
GFS_N = len(GFS_TIMES)
print(
    f"gfs: {GFS_N}/{len(D1_TIMES)} timestamps complete "
    f"({100 * GFS_N / len(D1_TIMES):.1f}% of window)"
)

In [ ]:
merra2_forecast_dir = "data/merra2"
merra2_existing = [f"{merra2_forecast_dir}/{name}" for name in os.listdir(merra2_forecast_dir)]
merra2_incomplete = set(
    pcr.pending_times(D1_TIMES, pcr.PCR_FLIGHT_LEVELS, merra2_existing, merra2_forecast_dir)
)
MERRA2_TIMES = sorted(t for t in D1_TIMES if t not in merra2_incomplete)
MERRA2_TIMES_SET = set(MERRA2_TIMES)
MERRA2_N = len(MERRA2_TIMES)
print(
    f"merra2: {MERRA2_N}/{len(D1_TIMES)} timestamps complete "
    f"({100 * MERRA2_N / len(D1_TIMES):.1f}% of window)"
)

In [ ]:
def filter_sepdec(df):
    return df[df["time"].isin(D1_TIMES_SET)]


def restrict_fl(df):
    """Restrict to the 14 flight levels the PCR family covers.

    Applied to contrails-org/google before the cross-family comparison so all
    four arms are scored over the same observation sample -- without this,
    contrails-org/google aggregate over their native FL270-440 while
    metoffice/era5 are structurally restricted to FL310-440, which is a
    different-sample bug, not a fair comparison.
    """
    return df[df["flight_level"].isin(pcr.PCR_FLIGHT_LEVELS)]


def filter_era5n(df):
    return df[df["time"].isin(ERA5_TIMES_SET)]


def filter_gfsn(df):
    return df[df["time"].isin(GFS_TIMES_SET)]


def filter_merra2n(df):
    return df[df["time"].isin(MERRA2_TIMES_SET)]


metoffice_purple = "#6a3d9a"
era5_green = "#1b9e77"
gfs_blue = "#377eb8"
merra2_gold = "#e6ab02"
contrails_org_amber = "#d95f02"
google_magenta = "#e7298a"

ARM_STYLE = {
    "metoffice": {"color": metoffice_purple, "ls": "-", "label": "Met Office"},
    "era5": {"color": era5_green, "ls": "-", "label": "ERA5"},
    "gfs": {"color": gfs_blue, "ls": "-", "label": "GFS"},
    "merra2": {"color": merra2_gold, "ls": "-", "label": "MERRA-2"},
    "contrails-org": {"color": contrails_org_amber, "ls": "--", "label": "Contrails.org"},
    "google": {"color": google_magenta, "ls": "--", "label": "Google"},
}

## Met Office (rhi_threshold sweep, full window, N=2928)

#### Final processing + bootstrapping

In [ ]:
df_adsb = open_dfs("metoffice-adsb-contrailwatch-region")
df_iagos = open_dfs("metoffice-iagos-contrailwatch-region")
df_gruan = open_dfs("metoffice-gruan-contrailwatch-region")
df_contrailwatch = open_dfs("metoffice-contrailwatch-contrailwatch-region")

In [ ]:
df = pd.DataFrame(
    {
        "penalty": penalty(df_adsb, "rhi_threshold"),
        "iagos_hit_rate": hit_rate(df_iagos, "rhi_threshold"),
        "gruan_hit_rate": hit_rate(df_gruan, "rhi_threshold"),
        "contrailwatch_hit_rate": hit_rate(df_contrailwatch, "rhi_threshold"),
    }
)

In [ ]:
rng = np.random.default_rng(301217)
n = 2_000
penalty_ci = bootstrap(
    df_adsb, df_adsb["time"].dt.date, n, lambda df: penalty(df, "rhi_threshold"), 0.05, rng
)
iagos_ci = bootstrap(
    df_iagos,
    df_iagos["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)
gruan_ci = bootstrap(
    df_gruan,
    df_gruan["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)
contrailwatch_ci = bootstrap(
    df_contrailwatch,
    df_contrailwatch["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)

In [ ]:
df_ci = pd.DataFrame(
    {
        "penalty_lo": penalty_ci.iloc[:, 0],
        "penalty_hi": penalty_ci.iloc[:, 1],
        "iagos_hit_rate_lo": iagos_ci.iloc[:, 0],
        "iagos_hit_rate_hi": iagos_ci.iloc[:, 1],
        "gruan_hit_rate_lo": gruan_ci.iloc[:, 0],
        "gruan_hit_rate_hi": gruan_ci.iloc[:, 1],
        "contrailwatch_hit_rate_lo": contrailwatch_ci.iloc[:, 0],
        "contrailwatch_hit_rate_hi": contrailwatch_ci.iloc[:, 1],
    }
)

In [ ]:
os.makedirs("data/benchmarks-local", exist_ok=True)
df.to_parquet("data/benchmarks-local/conus-sepdec-metoffice.pq")
df_ci.to_parquet("data/benchmarks-local/conus-sepdec-metoffice-ci.pq")

## Met Office (rhi_threshold sweep, era5's N subset — subsampling-error comparison)

#### Final processing + bootstrapping

In [ ]:
df_adsb = filter_era5n(open_dfs("metoffice-adsb-contrailwatch-region"))
df_iagos = filter_era5n(open_dfs("metoffice-iagos-contrailwatch-region"))
df_gruan = filter_era5n(open_dfs("metoffice-gruan-contrailwatch-region"))
df_contrailwatch = filter_era5n(open_dfs("metoffice-contrailwatch-contrailwatch-region"))

In [ ]:
df = pd.DataFrame(
    {
        "penalty": penalty(df_adsb, "rhi_threshold"),
        "iagos_hit_rate": hit_rate(df_iagos, "rhi_threshold"),
        "gruan_hit_rate": hit_rate(df_gruan, "rhi_threshold"),
        "contrailwatch_hit_rate": hit_rate(df_contrailwatch, "rhi_threshold"),
    }
)

In [ ]:
rng = np.random.default_rng(301217)
n = 2_000
penalty_ci = bootstrap(
    df_adsb, df_adsb["time"].dt.date, n, lambda df: penalty(df, "rhi_threshold"), 0.05, rng
)
iagos_ci = bootstrap(
    df_iagos,
    df_iagos["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)
gruan_ci = bootstrap(
    df_gruan,
    df_gruan["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)
contrailwatch_ci = bootstrap(
    df_contrailwatch,
    df_contrailwatch["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)

In [ ]:
df_ci = pd.DataFrame(
    {
        "penalty_lo": penalty_ci.iloc[:, 0],
        "penalty_hi": penalty_ci.iloc[:, 1],
        "iagos_hit_rate_lo": iagos_ci.iloc[:, 0],
        "iagos_hit_rate_hi": iagos_ci.iloc[:, 1],
        "gruan_hit_rate_lo": gruan_ci.iloc[:, 0],
        "gruan_hit_rate_hi": gruan_ci.iloc[:, 1],
        "contrailwatch_hit_rate_lo": contrailwatch_ci.iloc[:, 0],
        "contrailwatch_hit_rate_hi": contrailwatch_ci.iloc[:, 1],
    }
)

In [ ]:
os.makedirs("data/benchmarks-local", exist_ok=True)
df.to_parquet("data/benchmarks-local/conus-era5n-metoffice.pq")
df_ci.to_parquet("data/benchmarks-local/conus-era5n-metoffice-ci.pq")

## Met Office (rhi_threshold sweep, gfs's N subset — subsampling-error comparison)

#### Final processing + bootstrapping

In [ ]:
df_adsb = filter_gfsn(open_dfs("metoffice-adsb-contrailwatch-region"))
df_iagos = filter_gfsn(open_dfs("metoffice-iagos-contrailwatch-region"))
df_gruan = filter_gfsn(open_dfs("metoffice-gruan-contrailwatch-region"))
df_contrailwatch = filter_gfsn(open_dfs("metoffice-contrailwatch-contrailwatch-region"))

In [ ]:
df = pd.DataFrame(
    {
        "penalty": penalty(df_adsb, "rhi_threshold"),
        "iagos_hit_rate": hit_rate(df_iagos, "rhi_threshold"),
        "gruan_hit_rate": hit_rate(df_gruan, "rhi_threshold"),
        "contrailwatch_hit_rate": hit_rate(df_contrailwatch, "rhi_threshold"),
    }
)

In [ ]:
rng = np.random.default_rng(301217)
n = 2_000
penalty_ci = bootstrap(
    df_adsb, df_adsb["time"].dt.date, n, lambda df: penalty(df, "rhi_threshold"), 0.05, rng
)
iagos_ci = bootstrap(
    df_iagos,
    df_iagos["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)
gruan_ci = bootstrap(
    df_gruan,
    df_gruan["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)
contrailwatch_ci = bootstrap(
    df_contrailwatch,
    df_contrailwatch["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)

In [ ]:
df_ci = pd.DataFrame(
    {
        "penalty_lo": penalty_ci.iloc[:, 0],
        "penalty_hi": penalty_ci.iloc[:, 1],
        "iagos_hit_rate_lo": iagos_ci.iloc[:, 0],
        "iagos_hit_rate_hi": iagos_ci.iloc[:, 1],
        "gruan_hit_rate_lo": gruan_ci.iloc[:, 0],
        "gruan_hit_rate_hi": gruan_ci.iloc[:, 1],
        "contrailwatch_hit_rate_lo": contrailwatch_ci.iloc[:, 0],
        "contrailwatch_hit_rate_hi": contrailwatch_ci.iloc[:, 1],
    }
)

In [ ]:
os.makedirs("data/benchmarks-local", exist_ok=True)
df.to_parquet("data/benchmarks-local/conus-gfsn-metoffice.pq")
df_ci.to_parquet("data/benchmarks-local/conus-gfsn-metoffice-ci.pq")

## Met Office (rhi_threshold sweep, merra2's N subset — subsampling-error comparison)

#### Final processing + bootstrapping

In [ ]:
df_adsb = filter_merra2n(open_dfs("metoffice-adsb-contrailwatch-region"))
df_iagos = filter_merra2n(open_dfs("metoffice-iagos-contrailwatch-region"))
df_gruan = filter_merra2n(open_dfs("metoffice-gruan-contrailwatch-region"))
df_contrailwatch = filter_merra2n(open_dfs("metoffice-contrailwatch-contrailwatch-region"))

In [ ]:
df = pd.DataFrame(
    {
        "penalty": penalty(df_adsb, "rhi_threshold"),
        "iagos_hit_rate": hit_rate(df_iagos, "rhi_threshold"),
        "gruan_hit_rate": hit_rate(df_gruan, "rhi_threshold"),
        "contrailwatch_hit_rate": hit_rate(df_contrailwatch, "rhi_threshold"),
    }
)

In [ ]:
rng = np.random.default_rng(301217)
n = 2_000
penalty_ci = bootstrap(
    df_adsb, df_adsb["time"].dt.date, n, lambda df: penalty(df, "rhi_threshold"), 0.05, rng
)
iagos_ci = bootstrap(
    df_iagos,
    df_iagos["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)
gruan_ci = bootstrap(
    df_gruan,
    df_gruan["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)
contrailwatch_ci = bootstrap(
    df_contrailwatch,
    df_contrailwatch["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)

In [ ]:
df_ci = pd.DataFrame(
    {
        "penalty_lo": penalty_ci.iloc[:, 0],
        "penalty_hi": penalty_ci.iloc[:, 1],
        "iagos_hit_rate_lo": iagos_ci.iloc[:, 0],
        "iagos_hit_rate_hi": iagos_ci.iloc[:, 1],
        "gruan_hit_rate_lo": gruan_ci.iloc[:, 0],
        "gruan_hit_rate_hi": gruan_ci.iloc[:, 1],
        "contrailwatch_hit_rate_lo": contrailwatch_ci.iloc[:, 0],
        "contrailwatch_hit_rate_hi": contrailwatch_ci.iloc[:, 1],
    }
)

In [ ]:
os.makedirs("data/benchmarks-local", exist_ok=True)
df.to_parquet("data/benchmarks-local/conus-merra2n-metoffice.pq")
df_ci.to_parquet("data/benchmarks-local/conus-merra2n-metoffice-ci.pq")

## ERA5 (rhi_threshold sweep, N = all completed timestamps)

#### Final processing + bootstrapping

In [ ]:
df_adsb = open_dfs("era5-adsb-contrailwatch-region")
df_iagos = open_dfs("era5-iagos-contrailwatch-region")
df_gruan = open_dfs("era5-gruan-contrailwatch-region")
df_contrailwatch = open_dfs("era5-contrailwatch-contrailwatch-region")

In [ ]:
df = pd.DataFrame(
    {
        "penalty": penalty(df_adsb, "rhi_threshold"),
        "iagos_hit_rate": hit_rate(df_iagos, "rhi_threshold"),
        "gruan_hit_rate": hit_rate(df_gruan, "rhi_threshold"),
        "contrailwatch_hit_rate": hit_rate(df_contrailwatch, "rhi_threshold"),
    }
)

In [ ]:
rng = np.random.default_rng(301217)
n = 2_000
penalty_ci = bootstrap(
    df_adsb, df_adsb["time"].dt.date, n, lambda df: penalty(df, "rhi_threshold"), 0.05, rng
)
iagos_ci = bootstrap(
    df_iagos,
    df_iagos["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)
gruan_ci = bootstrap(
    df_gruan,
    df_gruan["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)
contrailwatch_ci = bootstrap(
    df_contrailwatch,
    df_contrailwatch["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)

In [ ]:
df_ci = pd.DataFrame(
    {
        "penalty_lo": penalty_ci.iloc[:, 0],
        "penalty_hi": penalty_ci.iloc[:, 1],
        "iagos_hit_rate_lo": iagos_ci.iloc[:, 0],
        "iagos_hit_rate_hi": iagos_ci.iloc[:, 1],
        "gruan_hit_rate_lo": gruan_ci.iloc[:, 0],
        "gruan_hit_rate_hi": gruan_ci.iloc[:, 1],
        "contrailwatch_hit_rate_lo": contrailwatch_ci.iloc[:, 0],
        "contrailwatch_hit_rate_hi": contrailwatch_ci.iloc[:, 1],
    }
)

In [ ]:
os.makedirs("data/benchmarks-local", exist_ok=True)
df.to_parquet("data/benchmarks-local/conus-era5n-era5.pq")
df_ci.to_parquet("data/benchmarks-local/conus-era5n-era5-ci.pq")

## ERA5 (rhi_threshold sweep, gfs's N subset — subsampling-error comparison)

#### Final processing + bootstrapping

In [ ]:
df_adsb = filter_gfsn(open_dfs("era5-adsb-contrailwatch-region"))
df_iagos = filter_gfsn(open_dfs("era5-iagos-contrailwatch-region"))
df_gruan = filter_gfsn(open_dfs("era5-gruan-contrailwatch-region"))
df_contrailwatch = filter_gfsn(open_dfs("era5-contrailwatch-contrailwatch-region"))

In [ ]:
df = pd.DataFrame(
    {
        "penalty": penalty(df_adsb, "rhi_threshold"),
        "iagos_hit_rate": hit_rate(df_iagos, "rhi_threshold"),
        "gruan_hit_rate": hit_rate(df_gruan, "rhi_threshold"),
        "contrailwatch_hit_rate": hit_rate(df_contrailwatch, "rhi_threshold"),
    }
)

In [ ]:
rng = np.random.default_rng(301217)
n = 2_000
penalty_ci = bootstrap(
    df_adsb, df_adsb["time"].dt.date, n, lambda df: penalty(df, "rhi_threshold"), 0.05, rng
)
iagos_ci = bootstrap(
    df_iagos,
    df_iagos["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)
gruan_ci = bootstrap(
    df_gruan,
    df_gruan["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)
contrailwatch_ci = bootstrap(
    df_contrailwatch,
    df_contrailwatch["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)

In [ ]:
df_ci = pd.DataFrame(
    {
        "penalty_lo": penalty_ci.iloc[:, 0],
        "penalty_hi": penalty_ci.iloc[:, 1],
        "iagos_hit_rate_lo": iagos_ci.iloc[:, 0],
        "iagos_hit_rate_hi": iagos_ci.iloc[:, 1],
        "gruan_hit_rate_lo": gruan_ci.iloc[:, 0],
        "gruan_hit_rate_hi": gruan_ci.iloc[:, 1],
        "contrailwatch_hit_rate_lo": contrailwatch_ci.iloc[:, 0],
        "contrailwatch_hit_rate_hi": contrailwatch_ci.iloc[:, 1],
    }
)

In [ ]:
os.makedirs("data/benchmarks-local", exist_ok=True)
df.to_parquet("data/benchmarks-local/conus-gfsn-era5.pq")
df_ci.to_parquet("data/benchmarks-local/conus-gfsn-era5-ci.pq")

## ERA5 (rhi_threshold sweep, merra2's N subset — subsampling-error comparison)

#### Final processing + bootstrapping

In [ ]:
df_adsb = filter_merra2n(open_dfs("era5-adsb-contrailwatch-region"))
df_iagos = filter_merra2n(open_dfs("era5-iagos-contrailwatch-region"))
df_gruan = filter_merra2n(open_dfs("era5-gruan-contrailwatch-region"))
df_contrailwatch = filter_merra2n(open_dfs("era5-contrailwatch-contrailwatch-region"))

In [ ]:
df = pd.DataFrame(
    {
        "penalty": penalty(df_adsb, "rhi_threshold"),
        "iagos_hit_rate": hit_rate(df_iagos, "rhi_threshold"),
        "gruan_hit_rate": hit_rate(df_gruan, "rhi_threshold"),
        "contrailwatch_hit_rate": hit_rate(df_contrailwatch, "rhi_threshold"),
    }
)

In [ ]:
rng = np.random.default_rng(301217)
n = 2_000
penalty_ci = bootstrap(
    df_adsb, df_adsb["time"].dt.date, n, lambda df: penalty(df, "rhi_threshold"), 0.05, rng
)
iagos_ci = bootstrap(
    df_iagos,
    df_iagos["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)
gruan_ci = bootstrap(
    df_gruan,
    df_gruan["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)
contrailwatch_ci = bootstrap(
    df_contrailwatch,
    df_contrailwatch["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)

In [ ]:
df_ci = pd.DataFrame(
    {
        "penalty_lo": penalty_ci.iloc[:, 0],
        "penalty_hi": penalty_ci.iloc[:, 1],
        "iagos_hit_rate_lo": iagos_ci.iloc[:, 0],
        "iagos_hit_rate_hi": iagos_ci.iloc[:, 1],
        "gruan_hit_rate_lo": gruan_ci.iloc[:, 0],
        "gruan_hit_rate_hi": gruan_ci.iloc[:, 1],
        "contrailwatch_hit_rate_lo": contrailwatch_ci.iloc[:, 0],
        "contrailwatch_hit_rate_hi": contrailwatch_ci.iloc[:, 1],
    }
)

In [ ]:
os.makedirs("data/benchmarks-local", exist_ok=True)
df.to_parquet("data/benchmarks-local/conus-merra2n-era5.pq")
df_ci.to_parquet("data/benchmarks-local/conus-merra2n-era5-ci.pq")

## GFS (rhi_threshold sweep, N=300 — new forecast-family arm)

#### Final processing + bootstrapping

In [ ]:
df_adsb = open_dfs("gfs-adsb-contrailwatch-region")
df_iagos = open_dfs("gfs-iagos-contrailwatch-region")
df_gruan = open_dfs("gfs-gruan-contrailwatch-region")
df_contrailwatch = open_dfs("gfs-contrailwatch-contrailwatch-region")

In [ ]:
df = pd.DataFrame(
    {
        "penalty": penalty(df_adsb, "rhi_threshold"),
        "iagos_hit_rate": hit_rate(df_iagos, "rhi_threshold"),
        "gruan_hit_rate": hit_rate(df_gruan, "rhi_threshold"),
        "contrailwatch_hit_rate": hit_rate(df_contrailwatch, "rhi_threshold"),
    }
)

In [ ]:
rng = np.random.default_rng(301217)
n = 2_000
penalty_ci = bootstrap(
    df_adsb, df_adsb["time"].dt.date, n, lambda df: penalty(df, "rhi_threshold"), 0.05, rng
)
iagos_ci = bootstrap(
    df_iagos,
    df_iagos["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)
gruan_ci = bootstrap(
    df_gruan,
    df_gruan["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)
contrailwatch_ci = bootstrap(
    df_contrailwatch,
    df_contrailwatch["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)

In [ ]:
df_ci = pd.DataFrame(
    {
        "penalty_lo": penalty_ci.iloc[:, 0],
        "penalty_hi": penalty_ci.iloc[:, 1],
        "iagos_hit_rate_lo": iagos_ci.iloc[:, 0],
        "iagos_hit_rate_hi": iagos_ci.iloc[:, 1],
        "gruan_hit_rate_lo": gruan_ci.iloc[:, 0],
        "gruan_hit_rate_hi": gruan_ci.iloc[:, 1],
        "contrailwatch_hit_rate_lo": contrailwatch_ci.iloc[:, 0],
        "contrailwatch_hit_rate_hi": contrailwatch_ci.iloc[:, 1],
    }
)

In [ ]:
os.makedirs("data/benchmarks-local", exist_ok=True)
df.to_parquet("data/benchmarks-local/conus-gfsn-gfs.pq")
df_ci.to_parquet("data/benchmarks-local/conus-gfsn-gfs-ci.pq")

## GFS (rhi_threshold sweep, merra2's N subset — subsampling-error comparison)

#### Final processing + bootstrapping

In [ ]:
df_adsb = filter_merra2n(open_dfs("gfs-adsb-contrailwatch-region"))
df_iagos = filter_merra2n(open_dfs("gfs-iagos-contrailwatch-region"))
df_gruan = filter_merra2n(open_dfs("gfs-gruan-contrailwatch-region"))
df_contrailwatch = filter_merra2n(open_dfs("gfs-contrailwatch-contrailwatch-region"))

In [ ]:
df = pd.DataFrame(
    {
        "penalty": penalty(df_adsb, "rhi_threshold"),
        "iagos_hit_rate": hit_rate(df_iagos, "rhi_threshold"),
        "gruan_hit_rate": hit_rate(df_gruan, "rhi_threshold"),
        "contrailwatch_hit_rate": hit_rate(df_contrailwatch, "rhi_threshold"),
    }
)

In [ ]:
rng = np.random.default_rng(301217)
n = 2_000
penalty_ci = bootstrap(
    df_adsb, df_adsb["time"].dt.date, n, lambda df: penalty(df, "rhi_threshold"), 0.05, rng
)
iagos_ci = bootstrap(
    df_iagos,
    df_iagos["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)
gruan_ci = bootstrap(
    df_gruan,
    df_gruan["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)
contrailwatch_ci = bootstrap(
    df_contrailwatch,
    df_contrailwatch["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)

In [ ]:
df_ci = pd.DataFrame(
    {
        "penalty_lo": penalty_ci.iloc[:, 0],
        "penalty_hi": penalty_ci.iloc[:, 1],
        "iagos_hit_rate_lo": iagos_ci.iloc[:, 0],
        "iagos_hit_rate_hi": iagos_ci.iloc[:, 1],
        "gruan_hit_rate_lo": gruan_ci.iloc[:, 0],
        "gruan_hit_rate_hi": gruan_ci.iloc[:, 1],
        "contrailwatch_hit_rate_lo": contrailwatch_ci.iloc[:, 0],
        "contrailwatch_hit_rate_hi": contrailwatch_ci.iloc[:, 1],
    }
)

In [ ]:
os.makedirs("data/benchmarks-local", exist_ok=True)
df.to_parquet("data/benchmarks-local/conus-merra2n-gfs.pq")
df_ci.to_parquet("data/benchmarks-local/conus-merra2n-gfs-ci.pq")

## MERRA-2 (rhi_threshold sweep, N=300 — new forecast-family arm; native 3-hourly cadence)

#### Final processing + bootstrapping

In [ ]:
df_adsb = open_dfs("merra2-adsb-contrailwatch-region")
df_iagos = open_dfs("merra2-iagos-contrailwatch-region")
df_gruan = open_dfs("merra2-gruan-contrailwatch-region")
df_contrailwatch = open_dfs("merra2-contrailwatch-contrailwatch-region")

In [ ]:
df = pd.DataFrame(
    {
        "penalty": penalty(df_adsb, "rhi_threshold"),
        "iagos_hit_rate": hit_rate(df_iagos, "rhi_threshold"),
        "gruan_hit_rate": hit_rate(df_gruan, "rhi_threshold"),
        "contrailwatch_hit_rate": hit_rate(df_contrailwatch, "rhi_threshold"),
    }
)

In [ ]:
rng = np.random.default_rng(301217)
n = 2_000
penalty_ci = bootstrap(
    df_adsb, df_adsb["time"].dt.date, n, lambda df: penalty(df, "rhi_threshold"), 0.05, rng
)
iagos_ci = bootstrap(
    df_iagos,
    df_iagos["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)
gruan_ci = bootstrap(
    df_gruan,
    df_gruan["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)
contrailwatch_ci = bootstrap(
    df_contrailwatch,
    df_contrailwatch["time"].dt.date,
    n,
    lambda df: hit_rate(df, "rhi_threshold"),
    0.05,
    rng,
)

In [ ]:
df_ci = pd.DataFrame(
    {
        "penalty_lo": penalty_ci.iloc[:, 0],
        "penalty_hi": penalty_ci.iloc[:, 1],
        "iagos_hit_rate_lo": iagos_ci.iloc[:, 0],
        "iagos_hit_rate_hi": iagos_ci.iloc[:, 1],
        "gruan_hit_rate_lo": gruan_ci.iloc[:, 0],
        "gruan_hit_rate_hi": gruan_ci.iloc[:, 1],
        "contrailwatch_hit_rate_lo": contrailwatch_ci.iloc[:, 0],
        "contrailwatch_hit_rate_hi": contrailwatch_ci.iloc[:, 1],
    }
)

In [ ]:
os.makedirs("data/benchmarks-local", exist_ok=True)
df.to_parquet("data/benchmarks-local/conus-merra2n-merra2.pq")
df_ci.to_parquet("data/benchmarks-local/conus-merra2n-merra2-ci.pq")

## Contrails.org (full window, N=2928)

#### Final processing + bootstrapping

In [ ]:
df_adsb = filter_sepdec(restrict_fl(open_dfs("contrails-org-adsb-contrailwatch-region")))
df_iagos = filter_sepdec(restrict_fl(open_dfs("contrails-org-iagos-contrailwatch-region")))
df_gruan = filter_sepdec(restrict_fl(open_dfs("contrails-org-gruan-contrailwatch-region")))
df_contrailwatch = filter_sepdec(restrict_fl(open_dfs("contrails-org-contrailwatch-contrailwatch-region")))

In [ ]:
df = pd.DataFrame(
    {
        "penalty": penalty(df_adsb, "horizontal_buffer"),
        "iagos_hit_rate": hit_rate(df_iagos, "horizontal_buffer"),
        "gruan_hit_rate": hit_rate(df_gruan, "horizontal_buffer"),
        "contrailwatch_hit_rate": hit_rate(df_contrailwatch, "horizontal_buffer"),
    }
)

In [ ]:
rng = np.random.default_rng(301217)
n = 2_000
penalty_ci = bootstrap(
    df_adsb, df_adsb["time"].dt.date, n, lambda df: penalty(df, "horizontal_buffer"), 0.05, rng
)
iagos_ci = bootstrap(
    df_iagos,
    df_iagos["time"].dt.date,
    n,
    lambda df: hit_rate(df, "horizontal_buffer"),
    0.05,
    rng,
)
gruan_ci = bootstrap(
    df_gruan,
    df_gruan["time"].dt.date,
    n,
    lambda df: hit_rate(df, "horizontal_buffer"),
    0.05,
    rng,
)
contrailwatch_ci = bootstrap(
    df_contrailwatch,
    df_contrailwatch["time"].dt.date,
    n,
    lambda df: hit_rate(df, "horizontal_buffer"),
    0.05,
    rng,
)

In [ ]:
df_ci = pd.DataFrame(
    {
        "penalty_lo": penalty_ci.iloc[:, 0],
        "penalty_hi": penalty_ci.iloc[:, 1],
        "iagos_hit_rate_lo": iagos_ci.iloc[:, 0],
        "iagos_hit_rate_hi": iagos_ci.iloc[:, 1],
        "gruan_hit_rate_lo": gruan_ci.iloc[:, 0],
        "gruan_hit_rate_hi": gruan_ci.iloc[:, 1],
        "contrailwatch_hit_rate_lo": contrailwatch_ci.iloc[:, 0],
        "contrailwatch_hit_rate_hi": contrailwatch_ci.iloc[:, 1],
    }
)

In [ ]:
os.makedirs("data/benchmarks-local", exist_ok=True)
df.to_parquet("data/benchmarks-local/conus-sepdec-contrails-org.pq")
df_ci.to_parquet("data/benchmarks-local/conus-sepdec-contrails-org-ci.pq")

## Contrails.org (era5's N subset)

#### Final processing + bootstrapping

In [ ]:
df_adsb = filter_era5n(restrict_fl(open_dfs("contrails-org-adsb-contrailwatch-region")))
df_iagos = filter_era5n(restrict_fl(open_dfs("contrails-org-iagos-contrailwatch-region")))
df_gruan = filter_era5n(restrict_fl(open_dfs("contrails-org-gruan-contrailwatch-region")))
df_contrailwatch = filter_era5n(restrict_fl(open_dfs("contrails-org-contrailwatch-contrailwatch-region")))

In [ ]:
df = pd.DataFrame(
    {
        "penalty": penalty(df_adsb, "horizontal_buffer"),
        "iagos_hit_rate": hit_rate(df_iagos, "horizontal_buffer"),
        "gruan_hit_rate": hit_rate(df_gruan, "horizontal_buffer"),
        "contrailwatch_hit_rate": hit_rate(df_contrailwatch, "horizontal_buffer"),
    }
)

In [ ]:
rng = np.random.default_rng(301217)
n = 2_000
penalty_ci = bootstrap(
    df_adsb, df_adsb["time"].dt.date, n, lambda df: penalty(df, "horizontal_buffer"), 0.05, rng
)
iagos_ci = bootstrap(
    df_iagos,
    df_iagos["time"].dt.date,
    n,
    lambda df: hit_rate(df, "horizontal_buffer"),
    0.05,
    rng,
)
gruan_ci = bootstrap(
    df_gruan,
    df_gruan["time"].dt.date,
    n,
    lambda df: hit_rate(df, "horizontal_buffer"),
    0.05,
    rng,
)
contrailwatch_ci = bootstrap(
    df_contrailwatch,
    df_contrailwatch["time"].dt.date,
    n,
    lambda df: hit_rate(df, "horizontal_buffer"),
    0.05,
    rng,
)

In [ ]:
df_ci = pd.DataFrame(
    {
        "penalty_lo": penalty_ci.iloc[:, 0],
        "penalty_hi": penalty_ci.iloc[:, 1],
        "iagos_hit_rate_lo": iagos_ci.iloc[:, 0],
        "iagos_hit_rate_hi": iagos_ci.iloc[:, 1],
        "gruan_hit_rate_lo": gruan_ci.iloc[:, 0],
        "gruan_hit_rate_hi": gruan_ci.iloc[:, 1],
        "contrailwatch_hit_rate_lo": contrailwatch_ci.iloc[:, 0],
        "contrailwatch_hit_rate_hi": contrailwatch_ci.iloc[:, 1],
    }
)

In [ ]:
os.makedirs("data/benchmarks-local", exist_ok=True)
df.to_parquet("data/benchmarks-local/conus-era5n-contrails-org.pq")
df_ci.to_parquet("data/benchmarks-local/conus-era5n-contrails-org-ci.pq")

## Contrails.org (gfs's N subset)

#### Final processing + bootstrapping

In [ ]:
df_adsb = filter_gfsn(restrict_fl(open_dfs("contrails-org-adsb-contrailwatch-region")))
df_iagos = filter_gfsn(restrict_fl(open_dfs("contrails-org-iagos-contrailwatch-region")))
df_gruan = filter_gfsn(restrict_fl(open_dfs("contrails-org-gruan-contrailwatch-region")))
df_contrailwatch = filter_gfsn(restrict_fl(open_dfs("contrails-org-contrailwatch-contrailwatch-region")))

In [ ]:
df = pd.DataFrame(
    {
        "penalty": penalty(df_adsb, "horizontal_buffer"),
        "iagos_hit_rate": hit_rate(df_iagos, "horizontal_buffer"),
        "gruan_hit_rate": hit_rate(df_gruan, "horizontal_buffer"),
        "contrailwatch_hit_rate": hit_rate(df_contrailwatch, "horizontal_buffer"),
    }
)

In [ ]:
rng = np.random.default_rng(301217)
n = 2_000
penalty_ci = bootstrap(
    df_adsb, df_adsb["time"].dt.date, n, lambda df: penalty(df, "horizontal_buffer"), 0.05, rng
)
iagos_ci = bootstrap(
    df_iagos,
    df_iagos["time"].dt.date,
    n,
    lambda df: hit_rate(df, "horizontal_buffer"),
    0.05,
    rng,
)
gruan_ci = bootstrap(
    df_gruan,
    df_gruan["time"].dt.date,
    n,
    lambda df: hit_rate(df, "horizontal_buffer"),
    0.05,
    rng,
)
contrailwatch_ci = bootstrap(
    df_contrailwatch,
    df_contrailwatch["time"].dt.date,
    n,
    lambda df: hit_rate(df, "horizontal_buffer"),
    0.05,
    rng,
)

In [ ]:
df_ci = pd.DataFrame(
    {
        "penalty_lo": penalty_ci.iloc[:, 0],
        "penalty_hi": penalty_ci.iloc[:, 1],
        "iagos_hit_rate_lo": iagos_ci.iloc[:, 0],
        "iagos_hit_rate_hi": iagos_ci.iloc[:, 1],
        "gruan_hit_rate_lo": gruan_ci.iloc[:, 0],
        "gruan_hit_rate_hi": gruan_ci.iloc[:, 1],
        "contrailwatch_hit_rate_lo": contrailwatch_ci.iloc[:, 0],
        "contrailwatch_hit_rate_hi": contrailwatch_ci.iloc[:, 1],
    }
)

In [ ]:
os.makedirs("data/benchmarks-local", exist_ok=True)
df.to_parquet("data/benchmarks-local/conus-gfsn-contrails-org.pq")
df_ci.to_parquet("data/benchmarks-local/conus-gfsn-contrails-org-ci.pq")

## Contrails.org (merra2's N subset)

#### Final processing + bootstrapping

In [ ]:
df_adsb = filter_merra2n(restrict_fl(open_dfs("contrails-org-adsb-contrailwatch-region")))
df_iagos = filter_merra2n(restrict_fl(open_dfs("contrails-org-iagos-contrailwatch-region")))
df_gruan = filter_merra2n(restrict_fl(open_dfs("contrails-org-gruan-contrailwatch-region")))
df_contrailwatch = filter_merra2n(restrict_fl(open_dfs("contrails-org-contrailwatch-contrailwatch-region")))

In [ ]:
df = pd.DataFrame(
    {
        "penalty": penalty(df_adsb, "horizontal_buffer"),
        "iagos_hit_rate": hit_rate(df_iagos, "horizontal_buffer"),
        "gruan_hit_rate": hit_rate(df_gruan, "horizontal_buffer"),
        "contrailwatch_hit_rate": hit_rate(df_contrailwatch, "horizontal_buffer"),
    }
)

In [ ]:
rng = np.random.default_rng(301217)
n = 2_000
penalty_ci = bootstrap(
    df_adsb, df_adsb["time"].dt.date, n, lambda df: penalty(df, "horizontal_buffer"), 0.05, rng
)
iagos_ci = bootstrap(
    df_iagos,
    df_iagos["time"].dt.date,
    n,
    lambda df: hit_rate(df, "horizontal_buffer"),
    0.05,
    rng,
)
gruan_ci = bootstrap(
    df_gruan,
    df_gruan["time"].dt.date,
    n,
    lambda df: hit_rate(df, "horizontal_buffer"),
    0.05,
    rng,
)
contrailwatch_ci = bootstrap(
    df_contrailwatch,
    df_contrailwatch["time"].dt.date,
    n,
    lambda df: hit_rate(df, "horizontal_buffer"),
    0.05,
    rng,
)

In [ ]:
df_ci = pd.DataFrame(
    {
        "penalty_lo": penalty_ci.iloc[:, 0],
        "penalty_hi": penalty_ci.iloc[:, 1],
        "iagos_hit_rate_lo": iagos_ci.iloc[:, 0],
        "iagos_hit_rate_hi": iagos_ci.iloc[:, 1],
        "gruan_hit_rate_lo": gruan_ci.iloc[:, 0],
        "gruan_hit_rate_hi": gruan_ci.iloc[:, 1],
        "contrailwatch_hit_rate_lo": contrailwatch_ci.iloc[:, 0],
        "contrailwatch_hit_rate_hi": contrailwatch_ci.iloc[:, 1],
    }
)

In [ ]:
os.makedirs("data/benchmarks-local", exist_ok=True)
df.to_parquet("data/benchmarks-local/conus-merra2n-contrails-org.pq")
df_ci.to_parquet("data/benchmarks-local/conus-merra2n-contrails-org-ci.pq")

## Google (full window, N=2928)

#### Final processing + bootstrapping

In [ ]:
df_adsb = filter_sepdec(restrict_fl(open_dfs("google-adsb-contrailwatch-region")))
df_iagos = filter_sepdec(restrict_fl(open_dfs("google-iagos-contrailwatch-region")))
df_gruan = filter_sepdec(restrict_fl(open_dfs("google-gruan-contrailwatch-region")))
df_contrailwatch = filter_sepdec(restrict_fl(open_dfs("google-contrailwatch-contrailwatch-region")))

In [ ]:
df = pd.DataFrame(
    {
        "penalty": penalty(df_adsb, "probability_threshold"),
        "iagos_hit_rate": hit_rate(df_iagos, "probability_threshold"),
        "gruan_hit_rate": hit_rate(df_gruan, "probability_threshold"),
        "contrailwatch_hit_rate": hit_rate(df_contrailwatch, "probability_threshold"),
    }
)

In [ ]:
rng = np.random.default_rng(301217)
n = 2_000
penalty_ci = bootstrap(
    df_adsb, df_adsb["time"].dt.date, n, lambda df: penalty(df, "probability_threshold"), 0.05, rng
)
iagos_ci = bootstrap(
    df_iagos,
    df_iagos["time"].dt.date,
    n,
    lambda df: hit_rate(df, "probability_threshold"),
    0.05,
    rng,
)
gruan_ci = bootstrap(
    df_gruan,
    df_gruan["time"].dt.date,
    n,
    lambda df: hit_rate(df, "probability_threshold"),
    0.05,
    rng,
)
contrailwatch_ci = bootstrap(
    df_contrailwatch,
    df_contrailwatch["time"].dt.date,
    n,
    lambda df: hit_rate(df, "probability_threshold"),
    0.05,
    rng,
)

In [ ]:
df_ci = pd.DataFrame(
    {
        "penalty_lo": penalty_ci.iloc[:, 0],
        "penalty_hi": penalty_ci.iloc[:, 1],
        "iagos_hit_rate_lo": iagos_ci.iloc[:, 0],
        "iagos_hit_rate_hi": iagos_ci.iloc[:, 1],
        "gruan_hit_rate_lo": gruan_ci.iloc[:, 0],
        "gruan_hit_rate_hi": gruan_ci.iloc[:, 1],
        "contrailwatch_hit_rate_lo": contrailwatch_ci.iloc[:, 0],
        "contrailwatch_hit_rate_hi": contrailwatch_ci.iloc[:, 1],
    }
)

In [ ]:
os.makedirs("data/benchmarks-local", exist_ok=True)
df.to_parquet("data/benchmarks-local/conus-sepdec-google.pq")
df_ci.to_parquet("data/benchmarks-local/conus-sepdec-google-ci.pq")

## Google (era5's N subset)

#### Final processing + bootstrapping

In [ ]:
df_adsb = filter_era5n(restrict_fl(open_dfs("google-adsb-contrailwatch-region")))
df_iagos = filter_era5n(restrict_fl(open_dfs("google-iagos-contrailwatch-region")))
df_gruan = filter_era5n(restrict_fl(open_dfs("google-gruan-contrailwatch-region")))
df_contrailwatch = filter_era5n(restrict_fl(open_dfs("google-contrailwatch-contrailwatch-region")))

In [ ]:
df = pd.DataFrame(
    {
        "penalty": penalty(df_adsb, "probability_threshold"),
        "iagos_hit_rate": hit_rate(df_iagos, "probability_threshold"),
        "gruan_hit_rate": hit_rate(df_gruan, "probability_threshold"),
        "contrailwatch_hit_rate": hit_rate(df_contrailwatch, "probability_threshold"),
    }
)

In [ ]:
rng = np.random.default_rng(301217)
n = 2_000
penalty_ci = bootstrap(
    df_adsb, df_adsb["time"].dt.date, n, lambda df: penalty(df, "probability_threshold"), 0.05, rng
)
iagos_ci = bootstrap(
    df_iagos,
    df_iagos["time"].dt.date,
    n,
    lambda df: hit_rate(df, "probability_threshold"),
    0.05,
    rng,
)
gruan_ci = bootstrap(
    df_gruan,
    df_gruan["time"].dt.date,
    n,
    lambda df: hit_rate(df, "probability_threshold"),
    0.05,
    rng,
)
contrailwatch_ci = bootstrap(
    df_contrailwatch,
    df_contrailwatch["time"].dt.date,
    n,
    lambda df: hit_rate(df, "probability_threshold"),
    0.05,
    rng,
)

In [ ]:
df_ci = pd.DataFrame(
    {
        "penalty_lo": penalty_ci.iloc[:, 0],
        "penalty_hi": penalty_ci.iloc[:, 1],
        "iagos_hit_rate_lo": iagos_ci.iloc[:, 0],
        "iagos_hit_rate_hi": iagos_ci.iloc[:, 1],
        "gruan_hit_rate_lo": gruan_ci.iloc[:, 0],
        "gruan_hit_rate_hi": gruan_ci.iloc[:, 1],
        "contrailwatch_hit_rate_lo": contrailwatch_ci.iloc[:, 0],
        "contrailwatch_hit_rate_hi": contrailwatch_ci.iloc[:, 1],
    }
)

In [ ]:
os.makedirs("data/benchmarks-local", exist_ok=True)
df.to_parquet("data/benchmarks-local/conus-era5n-google.pq")
df_ci.to_parquet("data/benchmarks-local/conus-era5n-google-ci.pq")

## Google (gfs's N subset)

#### Final processing + bootstrapping

In [ ]:
df_adsb = filter_gfsn(restrict_fl(open_dfs("google-adsb-contrailwatch-region")))
df_iagos = filter_gfsn(restrict_fl(open_dfs("google-iagos-contrailwatch-region")))
df_gruan = filter_gfsn(restrict_fl(open_dfs("google-gruan-contrailwatch-region")))
df_contrailwatch = filter_gfsn(restrict_fl(open_dfs("google-contrailwatch-contrailwatch-region")))

In [ ]:
df = pd.DataFrame(
    {
        "penalty": penalty(df_adsb, "probability_threshold"),
        "iagos_hit_rate": hit_rate(df_iagos, "probability_threshold"),
        "gruan_hit_rate": hit_rate(df_gruan, "probability_threshold"),
        "contrailwatch_hit_rate": hit_rate(df_contrailwatch, "probability_threshold"),
    }
)

In [ ]:
rng = np.random.default_rng(301217)
n = 2_000
penalty_ci = bootstrap(
    df_adsb, df_adsb["time"].dt.date, n, lambda df: penalty(df, "probability_threshold"), 0.05, rng
)
iagos_ci = bootstrap(
    df_iagos,
    df_iagos["time"].dt.date,
    n,
    lambda df: hit_rate(df, "probability_threshold"),
    0.05,
    rng,
)
gruan_ci = bootstrap(
    df_gruan,
    df_gruan["time"].dt.date,
    n,
    lambda df: hit_rate(df, "probability_threshold"),
    0.05,
    rng,
)
contrailwatch_ci = bootstrap(
    df_contrailwatch,
    df_contrailwatch["time"].dt.date,
    n,
    lambda df: hit_rate(df, "probability_threshold"),
    0.05,
    rng,
)

In [ ]:
df_ci = pd.DataFrame(
    {
        "penalty_lo": penalty_ci.iloc[:, 0],
        "penalty_hi": penalty_ci.iloc[:, 1],
        "iagos_hit_rate_lo": iagos_ci.iloc[:, 0],
        "iagos_hit_rate_hi": iagos_ci.iloc[:, 1],
        "gruan_hit_rate_lo": gruan_ci.iloc[:, 0],
        "gruan_hit_rate_hi": gruan_ci.iloc[:, 1],
        "contrailwatch_hit_rate_lo": contrailwatch_ci.iloc[:, 0],
        "contrailwatch_hit_rate_hi": contrailwatch_ci.iloc[:, 1],
    }
)

In [ ]:
os.makedirs("data/benchmarks-local", exist_ok=True)
df.to_parquet("data/benchmarks-local/conus-gfsn-google.pq")
df_ci.to_parquet("data/benchmarks-local/conus-gfsn-google-ci.pq")

## Google (merra2's N subset)

#### Final processing + bootstrapping

In [ ]:
df_adsb = filter_merra2n(restrict_fl(open_dfs("google-adsb-contrailwatch-region")))
df_iagos = filter_merra2n(restrict_fl(open_dfs("google-iagos-contrailwatch-region")))
df_gruan = filter_merra2n(restrict_fl(open_dfs("google-gruan-contrailwatch-region")))
df_contrailwatch = filter_merra2n(restrict_fl(open_dfs("google-contrailwatch-contrailwatch-region")))

In [ ]:
df = pd.DataFrame(
    {
        "penalty": penalty(df_adsb, "probability_threshold"),
        "iagos_hit_rate": hit_rate(df_iagos, "probability_threshold"),
        "gruan_hit_rate": hit_rate(df_gruan, "probability_threshold"),
        "contrailwatch_hit_rate": hit_rate(df_contrailwatch, "probability_threshold"),
    }
)

In [ ]:
rng = np.random.default_rng(301217)
n = 2_000
penalty_ci = bootstrap(
    df_adsb, df_adsb["time"].dt.date, n, lambda df: penalty(df, "probability_threshold"), 0.05, rng
)
iagos_ci = bootstrap(
    df_iagos,
    df_iagos["time"].dt.date,
    n,
    lambda df: hit_rate(df, "probability_threshold"),
    0.05,
    rng,
)
gruan_ci = bootstrap(
    df_gruan,
    df_gruan["time"].dt.date,
    n,
    lambda df: hit_rate(df, "probability_threshold"),
    0.05,
    rng,
)
contrailwatch_ci = bootstrap(
    df_contrailwatch,
    df_contrailwatch["time"].dt.date,
    n,
    lambda df: hit_rate(df, "probability_threshold"),
    0.05,
    rng,
)

In [ ]:
df_ci = pd.DataFrame(
    {
        "penalty_lo": penalty_ci.iloc[:, 0],
        "penalty_hi": penalty_ci.iloc[:, 1],
        "iagos_hit_rate_lo": iagos_ci.iloc[:, 0],
        "iagos_hit_rate_hi": iagos_ci.iloc[:, 1],
        "gruan_hit_rate_lo": gruan_ci.iloc[:, 0],
        "gruan_hit_rate_hi": gruan_ci.iloc[:, 1],
        "contrailwatch_hit_rate_lo": contrailwatch_ci.iloc[:, 0],
        "contrailwatch_hit_rate_hi": contrailwatch_ci.iloc[:, 1],
    }
)

In [ ]:
os.makedirs("data/benchmarks-local", exist_ok=True)
df.to_parquet("data/benchmarks-local/conus-merra2n-google.pq")
df_ci.to_parquet("data/benchmarks-local/conus-merra2n-google-ci.pq")

# Plots

In [ ]:
def load_arm(stem):
    df = pd.read_parquet(f"data/benchmarks-local/{stem}.pq")
    df_ci = pd.read_parquet(f"data/benchmarks-local/{stem}-ci.pq")
    return df, df_ci


def plot_arm_on_axis(ax, arm, df, df_ci, hit_rate_col):
    style = ARM_STYLE[arm]
    ax.plot(
        100 * df["penalty"],
        100 * df[hit_rate_col],
        marker="o",
        linestyle=style["ls"],
        color=style["color"],
        label=style["label"],
    )
    ax.plot(
        100 * df_ci[["penalty_lo", "penalty_hi"]].T,
        100 * df[[hit_rate_col, hit_rate_col]].T,
        linestyle=style["ls"],
        color=style["color"],
    )
    ax.plot(
        100 * df[["penalty", "penalty"]].T,
        100 * df_ci[[f"{hit_rate_col}_lo", f"{hit_rate_col}_hi"]].T,
        linestyle=style["ls"],
        color=style["color"],
    )


def plot_family_panel(ax, arms, hit_rate_col, obs_label):
    for arm, stem in arms:
        df, df_ci = load_arm(stem)
        plot_arm_on_axis(ax, arm, df, df_ci, hit_rate_col)
    ax.set_xlabel("Flight distance in forecast PCR (%)")
    ax.set_ylabel(f"{obs_label} hit rate (%)")
    ax.set_xlim([0, 30])
    ax.set_ylim([0, 100])
    ax.legend(loc="upper left", frameon=False)


ANNOTATION = (
    "Window: 1 Sep–31 Dec 2024, hourly, CONUS. RHi threshold swept 0.70–1.30 "
    "(step 0.05; metoffice/era5/gfs/merra2 only). Lead time: shortest available, T+0→T+5 "
    "(metoffice). Regrid target: existing benchmark grid. Solid = "
    "same-algorithm PCR arms; dashed = published products. All arms scored "
    "over FL310–440 (14 levels) for this comparison."
)

OBS_PANELS = [
    ("gruan_hit_rate", "GRUAN"),
    ("iagos_hit_rate", "IAGOS"),
    ("contrailwatch_hit_rate", "ContrailWatch"),
]

## Plot A — headline (metoffice / contrails-org / google, full window)

#### Plot

In [ ]:
PLOT_A_ARMS = [
    ("metoffice", "conus-sepdec-metoffice"),
    ("contrails-org", "conus-sepdec-contrails-org"),
    ("google", "conus-sepdec-google"),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax, (col, label) in zip(axes, OBS_PANELS):
    plot_family_panel(ax, PLOT_A_ARMS, col, label)

fig.suptitle(
    "Plot A — headline: metoffice vs contrails-org vs google "
    f"(CONUS, Sep–Dec 2024, N={len(D1_TIMES)}, detailed)",
    fontsize=12,
)
fig.text(0.5, -0.06, ANNOTATION, ha="center", fontsize=8, color=gray50)
plt.tight_layout()
plt.savefig("figures/conus_headline_detailed.png", dpi=300, bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
plot_family_panel(ax, PLOT_A_ARMS, "contrailwatch_hit_rate", "ContrailWatch")
ax.set_title(
    f"Plot A — headline (CONUS, Sep–Dec 2024, N={len(D1_TIMES)}, simplified)",
    fontsize=11,
)
fig.text(0.5, -0.18, ANNOTATION, ha="center", fontsize=7, color=gray50, wrap=True)
plt.tight_layout()
plt.savefig("figures/conus_headline_simple.png", dpi=300, bbox_inches="tight")

## Plot B — preview (all four arms, era5's N subset)

#### Plot

In [ ]:
PLOT_B_ARMS = [
    ("metoffice", "conus-era5n-metoffice"),
    ("era5", "conus-era5n-era5"),
    ("contrails-org", "conus-era5n-contrails-org"),
    ("google", "conus-era5n-google"),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax, (col, label) in zip(axes, OBS_PANELS):
    plot_family_panel(ax, PLOT_B_ARMS, col, label)

fig.suptitle(
    f"Plot B — PREVIEW: N={ERA5_N} of {len(D1_TIMES)} "
    f"({100 * ERA5_N / len(D1_TIMES):.1f}% of window), detailed",
    fontsize=12,
    color="firebrick",
)
fig.text(0.5, -0.06, ANNOTATION, ha="center", fontsize=8, color=gray50)
plt.tight_layout()
plt.savefig("figures/conus_era5n_preview_detailed.png", dpi=300, bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
plot_family_panel(ax, PLOT_B_ARMS, "contrailwatch_hit_rate", "ContrailWatch")
ax.set_title(
    f"Plot B — PREVIEW: N={ERA5_N} of {len(D1_TIMES)} "
    f"({100 * ERA5_N / len(D1_TIMES):.1f}% of window), simplified",
    fontsize=11,
    color="firebrick",
)
fig.text(0.5, -0.18, ANNOTATION, ha="center", fontsize=7, color=gray50, wrap=True)
plt.tight_layout()
plt.savefig("figures/conus_era5n_preview_simple.png", dpi=300, bbox_inches="tight")

## Plot C — preview (three-way PCR family + published products, gfs's N subset)

#### Plot

In [ ]:
PLOT_C_ARMS = [
    ("metoffice", "conus-gfsn-metoffice"),
    ("era5", "conus-gfsn-era5"),
    ("gfs", "conus-gfsn-gfs"),
    ("contrails-org", "conus-gfsn-contrails-org"),
    ("google", "conus-gfsn-google"),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax, (col, label) in zip(axes, OBS_PANELS):
    plot_family_panel(ax, PLOT_C_ARMS, col, label)

fig.suptitle(
    f"Plot C — PREVIEW: N={GFS_N} of {len(D1_TIMES)} "
    f"({100 * GFS_N / len(D1_TIMES):.1f}% of window), detailed",
    fontsize=12,
    color="firebrick",
)
fig.text(0.5, -0.06, ANNOTATION, ha="center", fontsize=8, color=gray50)
plt.tight_layout()
plt.savefig("figures/conus_gfsn_preview_detailed.png", dpi=300, bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
plot_family_panel(ax, PLOT_C_ARMS, "contrailwatch_hit_rate", "ContrailWatch")
ax.set_title(
    f"Plot C — PREVIEW: N={GFS_N} of {len(D1_TIMES)} "
    f"({100 * GFS_N / len(D1_TIMES):.1f}% of window), simplified",
    fontsize=11,
    color="firebrick",
)
fig.text(0.5, -0.18, ANNOTATION, ha="center", fontsize=7, color=gray50, wrap=True)
plt.tight_layout()
plt.savefig("figures/conus_gfsn_preview_simple.png", dpi=300, bbox_inches="tight")

## Plot D — preview (four-way PCR family + published products, merra2's N subset)

#### Plot

In [ ]:
PLOT_D_ARMS = [
    ("metoffice", "conus-merra2n-metoffice"),
    ("era5", "conus-merra2n-era5"),
    ("gfs", "conus-merra2n-gfs"),
    ("merra2", "conus-merra2n-merra2"),
    ("contrails-org", "conus-merra2n-contrails-org"),
    ("google", "conus-merra2n-google"),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax, (col, label) in zip(axes, OBS_PANELS):
    plot_family_panel(ax, PLOT_D_ARMS, col, label)

fig.suptitle(
    f"Plot D — PREVIEW: N={MERRA2_N} of {len(D1_TIMES)} "
    f"({100 * MERRA2_N / len(D1_TIMES):.1f}% of window), detailed",
    fontsize=12,
    color="firebrick",
)
fig.text(0.5, -0.06, ANNOTATION, ha="center", fontsize=8, color=gray50)
plt.tight_layout()
plt.savefig("figures/conus_merra2n_preview_detailed.png", dpi=300, bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
plot_family_panel(ax, PLOT_D_ARMS, "contrailwatch_hit_rate", "ContrailWatch")
ax.set_title(
    f"Plot D — PREVIEW: N={MERRA2_N} of {len(D1_TIMES)} "
    f"({100 * MERRA2_N / len(D1_TIMES):.1f}% of window), simplified",
    fontsize=11,
    color="firebrick",
)
fig.text(0.5, -0.18, ANNOTATION, ha="center", fontsize=7, color=gray50, wrap=True)
plt.tight_layout()
plt.savefig("figures/conus_merra2n_preview_simple.png", dpi=300, bbox_inches="tight")

## Section A — Met Office vs ERA5: the RHi-threshold comparison

**What was tested:** the RHi-threshold sweep (0.70-1.30, step 0.05, buffer fixed at 0) for both new forecast arms, over the ERA5-matched subset (N=653/2928 timestamps — the fraction ERA5's mirror had completed at the time of this run; `sepdec-era5n` in the table below). The metric is `penalty` — the fraction of flight distance falling inside the forecast PCR, i.e. how much airspace each arm flags.

In [ ]:
df_ab = pd.read_csv("../v1/benchmarks.csv")
sub = df_ab[(df_ab["region"] == "conus") & (df_ab["season"] == "sepdec-era5n")
            & (df_ab["forecast"].isin(["metoffice", "era5"])) & (df_ab["rhi_threshold"].notna())]
table_a = sub.pivot(index="rhi_threshold", columns="forecast", values="penalty").sort_index()
table_a["metoffice / era5 ratio"] = table_a["metoffice"] / table_a["era5"]
print(table_a.to_string(float_format=lambda x: f"{x:.4f}"))
print()
r = table_a.loc[1.10]
print(f"At RHi >= 1.10: Met Office retains {r['metoffice']:.4f} of flight distance flagged vs "
      f"ERA5's {r['era5']:.4f} -- a {r['metoffice/era5 ratio' if 'metoffice/era5 ratio' in r else 'metoffice / era5 ratio']:.2f}x ratio.")

**Caveat:** ERA5 is a reanalysis (a best estimate of the past atmospheric state, informed by observations), not an operational forecast like the Met Office UM output used everywhere else in this comparison — this is not evidence that Met Office's *forecasts* beat ERA5's *analyses*. The gap is a structural artifact of each product's supersaturation representation: the IFS (which underlies ERA5) clips relative humidity over ice back to 100% wherever ice cloud already exists in a gridbox, so ERA5 has very little RHi signal left above 1.0 to threshold on. The UM's own ceiling reaches ~1.65, so Met Office's higher thresholds still spread out properly.

## Section B — the observation-type reversal: ContrailWatch vs IAGOS

**What was tested:** at matched flight-distance-penalty operating points (Met Office swept over `rhi_threshold`, Google over `probability_threshold` — different knobs, so points are matched by *penalty*, not by the raw threshold value; Google's curve is linearly interpolated onto Met Office's own penalty grid), does the ranking between the two forecast sources agree across observation types? Full window (`sepdec`, N=2928 hourly timestamps, CONUS).

In [ ]:
df_b = pd.read_csv("../v1/benchmarks.csv")
mo_b = df_b[(df_b["region"] == "conus") & (df_b["season"] == "sepdec") & (df_b["forecast"] == "metoffice")].sort_values("penalty")
gg_b = df_b[(df_b["region"] == "conus") & (df_b["season"] == "sepdec") & (df_b["forecast"] == "google")].sort_values("penalty")
gp_b = gg_b["penalty"].to_numpy()

rows = []
for obs in ["contrailwatch_hit_rate", "iagos_hit_rate"]:
    g_lo, g_hi, g_mid = gg_b[f"{obs}_lo"].to_numpy(), gg_b[f"{obs}_hi"].to_numpy(), gg_b[obs].to_numpy()
    for _, row in mo_b.iterrows():
        p = row["penalty"]
        if p < gp_b.min() or p > gp_b.max():
            continue
        g_hi_i, g_lo_i, g_mid_i = np.interp(p, gp_b, g_hi), np.interp(p, gp_b, g_lo), np.interp(p, gp_b, g_mid)
        mo_lo, mo_hi, mo_mid = row[f"{obs}_lo"], row[f"{obs}_hi"], row[obs]
        if g_hi_i < mo_lo:
            verdict = "Met Office clearly above (CIs do not overlap)"
        elif mo_hi < g_lo_i:
            verdict = "Google clearly above (CIs do not overlap)"
        else:
            verdict = "overlap (not distinguishable at 95%)"
        rows.append({"obs_type": obs, "rhi_threshold": row["rhi_threshold"], "penalty": p,
                     "metoffice": f"[{mo_lo:.3f}, {mo_mid:.3f}, {mo_hi:.3f}]",
                     "google_at_matched_penalty": f"[{g_lo_i:.3f}, {g_mid_i:.3f}, {g_hi_i:.3f}]",
                     "verdict": verdict})
table_b = pd.DataFrame(rows)
pd.set_option("display.width", 160)
print(table_b.to_string(index=False))

**Caveat:** Google's ContrailWatch-validated hit rate is clearly above Met Office's at *every* matched-penalty point in this sweep — 95% CIs don't overlap anywhere in the table. On IAGOS the ranking reverses at the higher-penalty end of the sweep (`rhi_threshold` ≤ 0.75, penalty ≥ ~15%): Met Office's IAGOS hit-rate lower CI bound sits above Google's IAGOS hit-rate upper CI bound — a real, statistically clean reversal there, not a point-estimate wobble. At lower penalties (`rhi_threshold` ≥ 0.80) the IAGOS CIs still overlap, so the reversal is confined to the higher-penalty end of the currently-swept range, not the whole curve — extending the sweep to lower thresholds (below 0.70, not yet run) would be needed to know whether it holds more broadly.

## Section C — a flight-level correction, and the RHi=1.0 (ice-saturation) parity line

**What was tested:** two things. **(1)** A flight-level fix: `contrails-org`/`google` were being aggregated over their native 18 flight levels (FL270-440) in the cross-family comparison while `metoffice`/`era5` are structurally restricted to 14 (FL310-440) — restricting all four arms to the matched 14-level set (`restrict_fl`, applied above) is the correction; before/after is re-derived here directly from the raw per-hour `contrails-org` files at `horizontal_buffer=0`. **(2)** After that correction, at which RHi threshold does Met Office's own raw hit rate first reach Contrails.org's baseline (`buffer=0`) hit rate?

In [ ]:
import glob as _glob

def _open_dfs_v1(path):
    return pd.concat([pd.read_parquet(f) for f in sorted(_glob.glob(f"../v1/data/{path}/*.pq"))], axis="index")

_D1_START, _D1_END = pd.Timestamp("2024-09-01"), pd.Timestamp("2024-12-31 23:00")
_adsb_c = _open_dfs_v1("contrails-org-adsb-contrailwatch-region")
_iagos_c = _open_dfs_v1("contrails-org-iagos-contrailwatch-region")
_cw_c = _open_dfs_v1("contrails-org-contrailwatch-contrailwatch-region")
_adsb_c = _adsb_c[(_adsb_c["time"] >= _D1_START) & (_adsb_c["time"] <= _D1_END)]
_iagos_c = _iagos_c[(_iagos_c["time"] >= _D1_START) & (_iagos_c["time"] <= _D1_END)]
_cw_c = _cw_c[(_cw_c["time"] >= _D1_START) & (_cw_c["time"] <= _D1_END)]

before_after = []
for label, transform in [("before (native FL270-440, mismatched flight levels)", lambda d: d),
                          ("after (restricted FL310-440, the fix)", restrict_fl)]:
    a, i, c = transform(_adsb_c), transform(_iagos_c), transform(_cw_c)
    before_after.append({
        "state": label,
        "penalty": penalty(a, "horizontal_buffer").loc[0.0],
        "iagos_hit_rate": hit_rate(i, "horizontal_buffer").loc[0.0],
        "contrailwatch_hit_rate": hit_rate(c, "horizontal_buffer").loc[0.0],
    })
table_c1 = pd.DataFrame(before_after).set_index("state")
print(table_c1.to_string(float_format=lambda x: f"{x:.4f}"))
print()

# (2) parity crossing: where does metoffice's raw hit rate reach contrails-org's buffer=0 baseline?
df_c = pd.read_csv("../v1/benchmarks.csv")
mo_c = df_c[(df_c["region"] == "conus") & (df_c["season"] == "sepdec") & (df_c["forecast"] == "metoffice")].sort_values("rhi_threshold")
co0 = df_c[(df_c["region"] == "conus") & (df_c["season"] == "sepdec") & (df_c["forecast"] == "contrails-org")
           & (df_c["horizontal_buffer"] == 0)].iloc[0]
for col in ["contrailwatch_hit_rate", "iagos_hit_rate"]:
    rhi_arr, hit_arr = mo_c["rhi_threshold"].to_numpy(), mo_c[col].to_numpy()
    order = np.argsort(hit_arr)
    crossing = np.interp(co0[col], hit_arr[order], rhi_arr[order])
    print(f"{col}: Contrails.org baseline (buffer=0) = {co0[col]:.4f}; "
          f"Met Office reaches it at rhi_threshold ~= {crossing:.3f}")

**Caveat:** the correction itself is small — contrailwatch hit rate +1.4pp, IAGOS hit rate +0.07pp, penalty +0.6pp at `buffer=0` — it doesn't flip any ranking. Its value is making the cross-family comparison apples-to-apples (matched 14 flight levels) rather than resting on an 18-vs-14-level artifact. Separately: Met Office's own raw hit rate (both observation types) only reaches Contrails.org's simplest baseline (`buffer=0`) when its RHi threshold is set to roughly 0.96-0.97 or below — just under RHi=1.0, the ice-saturation line. At and above RHi=1.0, in the range where a Met Office operator would plausibly want to sit for a higher-confidence signal, it has not yet reached even Contrails.org's most conservative published operating point.